# Inverse Power Method

Inverse iteration targets the eigenvalue closest to shift (zero by default), using a reused sparse LU or dense LU inner solve. Factorization time is included.

Attach only bcsstk17.mat using Add Input. MAT_PATH overrides discovery; no downloads are used. Sparse CSR CPU implementation; sequential sweeps or sparse reusable inner solves use NumPy/Numba/SciPy.


In [ ]:
import time
from pathlib import Path
import numpy as np
from scipy.io import loadmat
from scipy.sparse import csr_matrix, issparse

print("Sparse CPU implementation")


def data_path(name, path=None):
    if path is not None:
        p = Path(path)
        if not p.is_file():
            raise FileNotFoundError(p)
        return p
    matches = sorted(Path("/kaggle/input").rglob(f"{name}.mat"))
    if not matches:
        p = Path("data") / f"{name}.mat"
        if p.is_file():
            return p
    if len(matches) != 1:
        raise FileNotFoundError(f"Attach {name}.mat or set MAT_PATH; matches: {matches}")
    return matches[0]



def relative_residual(A, x, b):
    return float(np.linalg.norm(A @ x - b) / max(np.linalg.norm(b), np.finfo(float).tiny))


def backward_substitution(U, b):
    x = np.zeros_like(b, dtype=np.float64)
    for i in range(len(b)-1, -1, -1):
        if U[i, i] == 0:
            raise ValueError("Zero pivot")
        x[i] = (b[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]
    return x


def lu_solve(L, U, b, p=None):
    c = np.array(b if p is None else b[p], dtype=np.float64, copy=True)
    for i in range(len(c)):
        c[i] = (c[i] - L[i, :i] @ c[:i]) / L[i, i]
    return backward_substitution(U, c)


def iteration_info(A, x, b, k, tol):
    r = relative_residual(A, x, b)
    return {"iterations": k, "residual": r, "converged": bool(np.isfinite(r) and r <= tol)}


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


In [ ]:
import numpy as np
from scipy.sparse import issparse, eye
from scipy.sparse.linalg import factorized
from scipy.linalg import lu_factor, lu_solve


def inverse_power_method(A, shift=0.0, x0=None, tol=1e-8, maxiter=10000):
    n = A.shape[0]
    if issparse(A):
        solve = factorized((A.astype(np.float64) - shift*eye(n)).tocsc())
    else:
        C = lu_factor(np.asarray(A, dtype=np.float64) - shift*np.eye(n))
        solve = lambda r: lu_solve(C, r)
    x = np.random.default_rng(42).normal(size=n) if x0 is None else np.array(x0, dtype=np.float64, copy=True)
    x /= np.linalg.norm(x)
    for k in range(maxiter+1):
        y = A @ x
        lam = float(x @ y)
        r = float(np.linalg.norm(y - lam*x) / max(np.linalg.norm(y), np.finfo(float).tiny))
        if r <= tol or k == maxiter:
            return lam, x, {"iterations": k, "residual": r, "converged": r <= tol}
        y = solve(x)
        if not np.all(np.isfinite(y)):
            raise ValueError("Singular shifted matrix")
        x = y / np.linalg.norm(y)


## 3×3


In [ ]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = inverse_power_method(A)
result, seconds = timed(inverse_power_method, A)
print("Seconds:", seconds)
lam, x, info = result
print("Eigenvalue:", lam)
print(info)
reference = np.linalg.eigvalsh(A)
print("Small reference eigenvalues:", reference)
assert info["converged"]
assert abs(lam - float(reference[0])) < 1e-6


## SuiteSparse: bcsstk17

Attach only bcsstk17.mat (10,974 × 10,974) using Add Input, or set MAT_PATH.

The matrix stays in sparse CSR storage. Inspect convergence status; the iteration limit may be reached.


In [ ]:
MATRIX = "bcsstk17"
MAT_PATH = None
A_raw = loadmat(data_path(MATRIX, MAT_PATH))["Problem"][0, 0]["A"].astype(np.float64)
A = csr_matrix(A_raw)
print("Matrix:", MATRIX, "shape:", A.shape)


In [ ]:
result, seconds = timed(inverse_power_method, A)
print("Seconds:", seconds)
lam, x, info = result
print("Eigenvalue:", lam)
print(info)
